# Learned donk and broky adapters: reproduce the first A100 experiment

Select **Runtime > Change runtime type > A100** before running. Actual first run used an A100-SXM4-80GB; training stopped at 600 updates, selecting step 400 from development loss. See the separately supplied validation report for measured results. Running this notebook starts a new experiment and consumes your Colab GPU allocation.

This freezes the pinned CS-NET v4 backbone and trains two identity/FiLM adapters. It does not install or control CS2, learn purchases, prove professional skill, or guarantee identical player quirks. Do not tune using final holdout results. The base checkpoint's original data overlap is unknown.

Data: Julien Blanchon / OpenCS2 (CC BY 4.0). Source licenses and exact starting points are retained in clone-model/THIRD_PARTY.md. The data archive includes one quarantined account; the pinned source games.json selects the verified nine matches only.


In [ ]:
import sys, subprocess, pathlib, torch, json, hashlib, urllib.request, zipfile
ROOT=pathlib.Path('/content/cs2-learned-clones-reproduction')
ROOT.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Connect an A100 GPU runtime first'
assert torch.cuda.get_device_properties(0).total_memory >= 32*1024**3, 'Use an A100-class GPU for batch 16'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
def streamed(args,cwd=None):
    process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait(): raise RuntimeError(f'Command failed: {args}')
streamed([sys.executable,'-m','pip','install','-q','webdataset','zstandard','open3d','demoparser2','pandas','pyyaml','huggingface_hub','pytest'])
for name,url,rev in [('csnet','https://github.com/Gary2005/cs-net.git','268dd2193c8a3d6e330c07dde5a851196c94eeaa'),('project','https://github.com/OhamjDung/HackUTA-CS2.git','99f850b')]:
    target=ROOT/name
    if not target.exists(): streamed(['git','clone',url,str(target)])
    streamed(['git','fetch','origin'],cwd=target)
    streamed(['git','checkout',rev],cwd=target)
streamed([sys.executable,'scripts/download_checkpoints.py','--pretrain-only','--out-dir',str(ROOT/'checkpoints')],cwd=ROOT/'csnet')


In [ ]:
archive=ROOT/'cs2-rounds-transfer.zip'
url='https://github.com/OhamjDung/HackUTA-CS2/releases/download/clone-training-data-v1/cs2-rounds-transfer.zip'
if not archive.exists(): urllib.request.urlretrieve(url,archive)
DATA_SHA='0e46914cbca514efbb5727b64f9e753116da4afaa98f764be66d15d97cb8ad25'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==DATA_SHA,'Data archive checksum mismatch'
with zipfile.ZipFile(archive) as z:
    assert all(not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts for n in z.namelist())
    z.extractall(ROOT)
GAMES=ROOT/'project/clone-model/games.json'
CACHE=ROOT/'cache-verified'
streamed([sys.executable,str(ROOT/'project/clone-model/prepare.py'),'--csnet',str(ROOT/'csnet'),'--json-dir',str(ROOT/'round_json'),'--games',str(GAMES),'--out',str(CACHE)])


In [ ]:
RUN=ROOT/'run'
RUN.mkdir(exist_ok=True)
common=['--csnet',str(ROOT/'csnet'),'--cache',str(CACHE),'--checkpoint',str(ROOT/'checkpoints/cs-net-v4-pro.pt'),'--out',str(RUN)]
config={'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'steps':1000,'batch_size':16,'lr':0.0003,'source_revision':'99f850b','data_sha256':DATA_SHA}
(RUN/'environment.json').write_text(json.dumps(config,indent=2))
streamed([sys.executable,str(ROOT/'project/clone-model/train.py'),*common,'--steps','1000','--batch-size','16','--device','cuda'])
streamed([sys.executable,str(ROOT/'project/clone-model/validate.py'),*common,'--adapter',str(RUN/'best.pt'),'--device','cuda'])
print(json.dumps(json.loads((RUN/'validation.json').read_text())['summary'],indent=2))
artifact=ROOT/'first-learned-clones.zip'
with zipfile.ZipFile(artifact,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.iterdir():
        if p.is_file():z.write(p,'run/'+p.name)
    z.write(CACHE/'manifest.json','manifest.json')
    z.write(GAMES,'games.json')
from google.colab import files
files.download(str(artifact))
